Alpha Leaf reduction setup

In [1]:
import pandas as pd
from hypernetx import Hypergraph

def build_hypergraph(edge_to_vertices):
    """
    Build X=(V,𝓔) from a hyperedge family 𝓔(X): edge_to_vertices[e] = set of vertices in e.
    Uses incidence table (v ∈ e) for stability.
    """
    rows = []
    for e, verts in edge_to_vertices.items():
        for v in verts:
            rows.append({"edges": e, "nodes": v, "weight": 1, "misc_properties": None})
    return Hypergraph(pd.DataFrame(rows))

def edge_sets(H):
    """Return 𝓔(H) as dict: e -> set(vertices in e)."""
    return {e: set(verts) for e, verts in H.incidence_dict.items()}

def print_hyperedges(H, title="Hyperedges"):
    print(title)
    for e, verts in edge_sets(H).items():
        print(f"  {e}: {sorted(verts)}")

def minimize_by_inclusion(H):
    """
    Safe minimization:
    If H has 0 or 1 hyperedges, return H.
    Otherwise, keep only inclusion-maximal hyperedges (toplexes).
    """
    E_map = edge_sets(H)
    if len(E_map) <= 1:
        return H
    return H.restrict_to_edges(H.toplexes())

def remove_size1_edges(H):
    """
    Enforce |e|>=2 for our setting.
    Safe on empty hypergraphs.
    """
    E_map = edge_sets(H)
    if not E_map:
        return H
    keep = [e for e, verts in E_map.items() if len(verts) >= 2]
    if not keep:
        # no hyperedges survive
        return H.restrict_to_edges([])  # produces an empty hypergraph object
    return H.restrict_to_edges(keep)

def vertex_incidence_degree(H):
    """deg_H(x) = number of hyperedges containing x."""
    deg = {}
    for e, verts in H.incidence_dict.items():
        for x in verts:
            deg[x] = deg.get(x, 0) + 1
    return deg

def alpha_leaves_and_parents(H):
    """
    α-leaf: deg=1
    parent: deg>=2
    """
    deg = vertex_incidence_degree(H)
    alpha = {x for x, d in deg.items() if d == 1}
    parents = {x for x, d in deg.items() if d >= 2}
    return alpha, parents, deg

def incident_hyperedges(H, x):
    """Return { e ∈ 𝓔(H) : x ∈ e }."""
    return [e for e, verts in H.incidence_dict.items() if x in verts]

WE BUILD ADJACENCY GRAPH BETWEEN INTERSECTING HYPEREDGES IN DUAL

In [2]:
def hyperedge_adjacency(E_map):
    """
    Build adjacency among hyperedges by nonempty intersection.
    Nodes: hyperedges (keys of E_map)
    Edge between e,f if E_map[e] ∩ E_map[f] ≠ ∅
    """
    edges = list(E_map.keys()) # list of hyperedges
    adj = {e: set() for e in edges} # dictionary of hyperedge -> set of its adjacent hyperedges
    for i in range(len(edges)): # loop over each hyperedge
        for j in range(i+1, len(edges)): # for each hyperedge, loop over subsequent hyperedges to avoid double counting
            e, f = edges[i], edges[j] # get the two hyperedges
            if E_map[e] & E_map[f]: # if they intersect (nonempty intersection)
                adj[e].add(f) # add f to e's adjacency set
                adj[f].add(e) # add e to f's adjacency set (undirected)
    return adj

def is_connected_hyperedge_graph(E_map):
    """
    Connectivity of the hyperedge-intersection graph.
    This is your 'are E1 and E3 still connected after deleting E2?' test.
    """
    if not E_map: # check if E_map is empty i.e., no hyperedges
        return True  # empty is vacuously connected for our purposes
    adj = hyperedge_adjacency(E_map) # get the adjacency list of the hyperedge graph
    start = next(iter(adj)) # get an arbitrary starting hyperedge default - (first key in the adjacency dictionary)
    seen = {start} # create a set to track seen hyperedges, initialized with the starting hyperedge
    stack = [start] # create a stack for searching, initialized with the starting hyperedge
    while stack: # while there are hyperedges to explore
        cur = stack.pop() # get the next hyperedge to explore0
        for nb in adj[cur]: # loop over the neighbors of the current hyperedge
            if nb not in seen: # if the neighbor hyperedge has not been seen yet
                seen.add(nb) # mark it as seen
                stack.append(nb) # add it to the stack for further exploration
    return len(seen) == len(adj) # check if all hyperedges were seen (i.e., the graph is connected)

Now we find the unique hyperedge an alpha belongs to

In [3]:
def unique_hyperedge_of_alpha_leaf(H, leaf):
    """
    α-leaf has exactly one incident hyperedge in M(H*).
    Return that hyperedge id.
    """
    inc = incident_hyperedges(H, leaf) # get the list of hyperedges that contain the leaf vertex
    if len(inc) != 1:
        raise ValueError(f"{leaf} is not an α-leaf (incident edges = {inc})")
    return inc[0]

We begin reduction attempt

In [4]:
def reduce_step(H_star_hat, clique_contents):
    """
    Attempt ONE safe reduction step on M(H*).
    Returns: (H_new, clique_contents_new, did_reduce, debug_info)
    """
    # get the hyperedge family as a dict: e -> set of vertices in e
    E_map = edge_sets(H_star_hat) 
    # get α-leaves, parents, and degree dict for vertices in H_star_hat
    alpha, parents, deg = alpha_leaves_and_parents(H_star_hat)

    # If no α-leaves, we cannot reduce since that means the graph is not helly
    if not alpha:
        return H_star_hat, clique_contents, False, {"reason": "no α-leaves"}

    # Pick an α-leaf (for now: deterministic choice by name)
    leaf = sorted(alpha, key=str)[0]
    # Find the unique hyperedge Ei that contains this α-leaf
    Ei = unique_hyperedge_of_alpha_leaf(H_star_hat, leaf)

    # Identify children in Ei = α-leaves contained in Ei
    children_in_Ei = sorted([x for x in E_map[Ei] if x in alpha], key=str)

    # Identify parents in Ei = vertices with deg>=2 contained in Ei
    parents_in_Ei = sorted([x for x in E_map[Ei] if x in parents], key=str)

    debug = {
        "chosen_leaf": leaf,
        "Ei": Ei,
        "children_in_Ei": children_in_Ei,
        "parents_in_Ei": parents_in_Ei,
        "case": None,
        "accepted": None
    }

    # If we delete Ei, will the remaining hyperedges still be connected?
    # we want to check if the leaf is good or bad
    # we create a a neew hyperedge family dict without Ei to check connectivity
    E_after_delete = {e: set(vs) for e, vs in E_map.items() if e != Ei}

    # we create the adjacency graph of the remaining hyperedges 
    # and check if it is connected
    good = is_connected_hyperedge_graph(E_after_delete)

    # if the check failed, then we know that the leaf is bad
    # and we reject this reduction step
    if not good:
        debug["accepted"] = False
        debug["reason"] = "deleting Ei disconnects hyperedge adjacency graph"
        return H_star_hat, clique_contents, False, debug

    # If it’s good, we ok the deletion of Ei plus parent-reduction, if needed
    # we create new clique contents dict to update for the reduction step
    clique_contents_new = {k: set(v) for k, v in clique_contents.items()}  # deep-ish copy

    # CASE R2: single child in Ei -> reduce ALL parents in Ei by removing leaf's content
    if len(children_in_Ei) == 1:
        debug["case"] = "single-child (R2: parent-content reduction)"

        # get the vertices that make the leaf clique 
        leaf_content = clique_contents_new.get(leaf, set())
        for p in parents_in_Ei:
            # Reduce parent clique-content: C(p) <- C(p) \ C(leaf)
            clique_contents_new[p] = clique_contents_new.get(p, set()) - leaf_content

        # (Cluster-control checks will happen AFTER recomputing H*, in a later function)
        # For now we only implement the core update + delete Ei.

    # CASE R1: multiple children in Ei -> do NOT reduce parents; just delete Ei
    else:
        debug["case"] = "multi-child (R1: no parent reduction)"

    # Delete Ei from hyperedge family
    H_new = H_star_hat.restrict_to_edges([e for e in E_map.keys() if e != Ei])

    # If no hyperedges remain, we're done; don't call toplexes()
    if len(edge_sets(H_new)) == 0:
        debug["accepted"] = True
        debug["reason"] = "all hyperedges deleted; stopping"
        return H_new, clique_contents_new, True, debug

    # Re-minimize and remove singletons again (your pipeline)
    H_new = minimize_by_inclusion(H_new)
    H_new = remove_size1_edges(H_new)

    debug["accepted"] = True
    return H_new, clique_contents_new, True, debug

TEST

In [5]:
# Toy H* hyperedges (already minimized + |e|>=2)
E_star_hat = {
    "E1": {"A","B","C","F"},
    "E2": {"C","E"}
}

# Toy clique contents (each clique-vertex corresponds to a maximal clique of G)
clique_contents = {
    "A": {"v1","v2","v3"},
    "B": {"v2","v4"},
    "C": {"v2","v5","v6"},
    "D": {"v5"},          # leaf-content
    "E": {"v6","v7"},
    "F": {"v8","v2"},
}

H_star_hat = build_hypergraph(E_star_hat)
H_star_hat = minimize_by_inclusion(H_star_hat)
H_star_hat = remove_size1_edges(H_star_hat)

print_hyperedges(H_star_hat, "Initial Ĥ*:")

for step in range(1, 6):
    H_star_hat, clique_contents, did, info = reduce_step(H_star_hat, clique_contents)
    print("\nStep", step, "| did_reduce =", did, "| info =", info)
    print_hyperedges(H_star_hat, "Ĥ* after step:")
    if not did:
        break

Initial Ĥ*:
  E1: ['A', 'B', 'C', 'F']
  E2: ['C', 'E']

Step 1 | did_reduce = True | info = {'chosen_leaf': 'A', 'Ei': 'E1', 'children_in_Ei': ['A', 'B', 'F'], 'parents_in_Ei': ['C'], 'case': 'multi-child (R1: no parent reduction)', 'accepted': True}
Ĥ* after step:
  E2: ['C', 'E']

Step 2 | did_reduce = True | info = {'chosen_leaf': 'C', 'Ei': 'E2', 'children_in_Ei': ['C', 'E'], 'parents_in_Ei': [], 'case': 'multi-child (R1: no parent reduction)', 'accepted': True, 'reason': 'all hyperedges deleted; stopping'}
Ĥ* after step:

Step 3 | did_reduce = False | info = {'reason': 'no α-leaves'}
Ĥ* after step:


Lets try adding Priority to reductions 

In [6]:
def classify_hyperedges_by_children(H_star_hat):
    """
    Return:
      - alpha: set of α-leaf vertices
      - parents: set of parent vertices
      - children_in_edge: dict e -> list of α-leaves in e
      - parents_in_edge:  dict e -> list of parents in e
    """
    E_map = edge_sets(H_star_hat)
    alpha, parents, deg = alpha_leaves_and_parents(H_star_hat)

    children_in_edge = {}
    parents_in_edge = {}
    for e, verts in E_map.items():
        children_in_edge[e] = sorted([x for x in verts if x in alpha], key=str)
        parents_in_edge[e]  = sorted([x for x in verts if x in parents], key=str)

    return alpha, parents, children_in_edge, parents_in_edge

We find the hyperedges with single childs as the preference

In [7]:
def pick_good_hyperedge_to_delete(H_star_hat, prefer_single_child=True):
    """
    Choose a hyperedge Ei to delete such that deleting Ei keeps the hyperedge-adjacency graph connected.
    Priority:
      - if prefer_single_child: try edges with exactly 1 child first
      - else: any good edge
    Returns: (Ei, case, children, parents) or (None, None, None, None)
      case = "single" or "multi"
    """
    E_map = edge_sets(H_star_hat)
    if len(E_map) <= 1:
        return None, None, None, None

    alpha, parents, children_in_edge, parents_in_edge = classify_hyperedges_by_children(H_star_hat)

    def is_good_delete(Ei):
        E_after = {e: set(vs) for e, vs in E_map.items() if e != Ei}
        return is_connected_hyperedge_graph(E_after)

    # Candidate lists
    single_candidates = [e for e in E_map if len(children_in_edge[e]) == 1]
    multi_candidates  = [e for e in E_map if len(children_in_edge[e]) >= 2]

    # Deterministic ordering (so runs are reproducible)
    single_candidates = sorted(single_candidates, key=str)
    multi_candidates  = sorted(multi_candidates, key=str)

    if prefer_single_child:
        for e in single_candidates:
            if is_good_delete(e):
                return e, "single", children_in_edge[e], parents_in_edge[e]
        for e in multi_candidates:
            if is_good_delete(e):
                return e, "multi", children_in_edge[e], parents_in_edge[e]
    else:
        for e in single_candidates + multi_candidates:
            if is_good_delete(e):
                return e, "single" if len(children_in_edge[e]) == 1 else "multi", children_in_edge[e], parents_in_edge[e]

    return None, None, None, None

In [8]:
def execute_delete_step(H_star_hat, clique_contents, Ei, case, children, parents):
    """
    Delete Ei from H* and apply the correct update rule based on the case.
    Returns updated (H_new, clique_contents_new).
    """
    E_map = edge_sets(H_star_hat)

    clique_contents_new = {k: set(v) for k, v in clique_contents.items()}

    if case == "single":
        # exactly one child leaf L in Ei
        L = children[0]
        L_content = clique_contents_new.get(L, set())

        # Reduce ALL parents in Ei (per your latest rule)
        for p in parents:
            clique_contents_new[p] = clique_contents_new.get(p, set()) - L_content

    # case == "multi": no parent reductions

    # delete Ei
    H_new = H_star_hat.restrict_to_edges([e for e in E_map.keys() if e != Ei])

    # re-minimize + remove singletons (safe versions)
    H_new = minimize_by_inclusion(H_new)
    H_new = remove_size1_edges(H_new)

    return H_new, clique_contents_new

In [9]:
def pick_one(S):
    """Deterministic choice from a set of G-vertices."""
    if not S:
        return None
    return sorted(S, key=str)[0]

Lets work on cluster control

In [10]:
def choose_cluster_parent_in_edge(E_map, Ei, x, deg):
    """
    x is an α-leaf in Ei. Choose C ∈ Ei\{x} as its parent for containment test.
    Prefer deg(C) >= 2; otherwise pick any other vertex deterministically.
    """
    # get all vertices in Ei except x
    candidates = [v for v in E_map[Ei] if v != x] 
    # if there are no candidates, return None 
    # (should not happen if Ei has at least 2 vertices)
    if not candidates:
        return None

    # among candidates, prefer those with degree >= 2
    strong = [v for v in candidates if deg.get(v, 0) >= 2]
    # if there are strong candidates, use them; otherwise use all candidates
    pool = strong if strong else candidates
    return sorted(pool, key=str)[0]

<>:2: DeprecationWarning: invalid escape sequence '\{'
<>:2: DeprecationWarning: invalid escape sequence '\{'
C:\Users\agney\AppData\Local\Temp\ipykernel_119952\1703729249.py:2: DeprecationWarning: invalid escape sequence '\{'
  """


In [11]:
def rebuild_from_edge_map(E_map):
    """
    PAPER: Build hypergraph from explicit hyperedge family 𝓔 (edge -> vertex set).
    """
    # Remove empty edges
    E_clean = {e: set(vs) for e, vs in E_map.items() if len(vs) > 0}
    if not E_clean:
        return build_hypergraph({})  # empty hypergraph
    H = build_hypergraph(E_clean)
    H = minimize_by_inclusion(H)
    H = remove_size1_edges(H)
    return H

In [12]:
def apply_cluster_control(H, clique_contents, reduced_parents):
    """
    Implements two-case rule on each reduced parent B:
      - if B becomes α-leaf and its chosen parent C satisfies C(B) ⊆ C(C), remove B
      - else do nothing
    Repeat until no more removals happen (cascade).
    """
    while True:
        # get the hyperedge family as a dict: e -> set(vertices in e)
        E_map = edge_sets(H)
        # If no hyperedges remain, we're done; don't call toplexes()
        # or do any further processing since the hypergraph is empty
        if len(E_map) == 0:
            return H

        # get α-leaves, parents, and degree dict for vertices in H
        alpha, parents, deg = alpha_leaves_and_parents(H)

        changed = False

        # Only check the vertices we just reduced (and still exist in H)
        # reduced_parents is the set of parents that 
        # we just reduced in the previous step, 
        # so we only check those for cluster control
        for B in sorted(set(reduced_parents), key=str):
            # check if the parent has becomee a leaf
            if B not in deg:
                continue  # B no longer exists in the hypergraph
            
            # If B is not an α-leaf, skip it; no cluster control needed
            if deg[B] != 1:
                continue  # Case 2: B remains a parent (or isolated); no cluster control

            # Case 1: B is α-leaf; find its unique incident hyperedge Ei
            inc = incident_hyperedges(H, B)
            # If B is an α-leaf, it should have exactly one incident hyperedge; 
            # if not, skip it (should not happen)
            if len(inc) != 1:
                continue
            Ei = inc[0]

            # find the parents in Ei 
            C = choose_cluster_parent_in_edge(E_map, Ei, B, deg)

            # if there are no parents to choose from, 
            # skip cluster control for this B, 
            # this should not happen since B is in Ei
            # and should have at least one other vertex as a parent
            if C is None:
                continue
            
            # get the verticees making up the cliques for B and C
            CB = clique_contents.get(B, set())
            CC = clique_contents.get(C, set())

            # check if the content of B is contained in the content of C
            if CB.issubset(CC):
                # remove B from Ei
                E_map[Ei] = set(E_map[Ei])
                E_map[Ei].discard(B)

                # rebuild + re-minimize + singleton-clean
                H = rebuild_from_edge_map(E_map)

                changed = True
                break  # restart, because degrees changed

        if not changed:
            return H

In [13]:
def base_case_cluster_control(H, clique_contents):
    """
    PAPER: When only one hyperedge E remains, remove redundant vertices inside E:
    if there exist x != y in E such that C(x) ⊆ C(y), delete x from E.
    Repeat until no more deletions.
    This prevents the last edge size from being artificially inflated, which affects movement base add.
    """
    E_map = edge_sets(H)
    if len(E_map) != 1:
        return H

    last_e = next(iter(E_map))
    E = set(E_map[last_e])

    changed = True
    while changed:
        changed = False
        verts = sorted(E, key=str)

        # try to find a dominated vertex x with a sibling y
        for i, x in enumerate(verts):
            Cx = clique_contents.get(x, set())
            for y in verts:
                if x == y:
                    continue
                Cy = clique_contents.get(y, set())
                if Cx.issubset(Cy):
                    # remove x
                    E.remove(x)
                    changed = True
                    break
            if changed:
                break

    # rebuild hypergraph with the cleaned last edge
    E_map[last_e] = E
    return rebuild_from_edge_map(E_map)

In [14]:
def execute_delete_step(H_star_hat, clique_contents, Ei, case, children, parents):
    """
    Delete Ei and apply correct update rule.
    If single-child: reduce ALL parents in Ei, then run cluster control.
    """
    # get the hyperedge family as a dict: e -> set(vertices in e)
    E_map = edge_sets(H_star_hat)
    # create a new clique contents dict to update for the reduction step
    clique_contents_new = {k: set(v) for k, v in clique_contents.items()}

    reduced_parents = []
    # CASE R2: single child in Ei 
    # -> reduce ALL parents in Ei by removing leaf's content
    if case == "single":
        # get the only child leaf L in Ei
        L = children[0] # children is a list of α-leaves in Ei; we know there's exactly one in this case
        # get the vertices that make the leaf clique
        L_content = clique_contents_new.get(L, set())
        # loop through every parent in the edge
        for p in parents:
            # get the clique content of the parent p, and reduce it by removing the leaf content
            clique_contents_new[p] = clique_contents_new.get(p, set()) - L_content
            # add the reduced parent to the list of reduced parents
            reduced_parents.append(p)

    # delete Ei
    H_new = H_star_hat.restrict_to_edges([e for e in E_map.keys() if e != Ei])
    H_new = minimize_by_inclusion(H_new)
    H_new = remove_size1_edges(H_new)

    # cluster control only matters after parent reductions (single-child case)
    # after you compute H_new and clique_contents_new
    if case == "single" and reduced_parents:
      H_new = apply_cluster_control(H_new, clique_contents_new, reduced_parents)

    return H_new, clique_contents_new

In [15]:
def single_child_guard_location(leaf, parents, clique_contents): #R2
    # get the clique content of the leaf vertex, which represents the vertices in G that correspond to this leaf
    # we take the leaf and return its contents as a set to CL
    CL = clique_contents.get(leaf, set())
    common = set() # Empty set
    # loop through the parents from the parents list
    for p in parents:
        # get the clique content of the parent p, which represents the vertices in G that correspond to this parent
        CP = clique_contents.get(p, set())
        # we take the intersection of the leaf content and the parent content, and add it to the common set
        # | means + but for sets. We are basically getting the common vertices in CL and Cp,
        # then we are adding them (or taking union of that subset) to the common set
        common |= (CL & CP)
        # we only need one of the common vertices to place the guard, so we return just one
    return pick_one(common)

In [16]:
def multi_child_guard_location(children, clique_contents): #R1
    # we give the list of children and the clique contents,
    # we then give sets the list of the clique contents of the children, 
    # which are sets of vertices in G corresponding to each child
    sets = [clique_contents.get(c, set()) for c in children]
    if not sets:
        return None
    # we want to take the intersection of the all the sets,
    # but we cannot do that without converting the sets list into a list of sets
    # so, we use map(set, sets) to apply sets() to each element if sets list,
    # which gives us a list of sets, and then we can take the intersection of all those sets using set.intersection.
    # However, the interesection function needs multiple individual sets not a list of sets, so we haave to unpack the list. 
    # To do this we should use the * operator to unpack the list of sets into 
    # individual arguments for set.intersection, which will give us the intersection of all the sets
    common = set.intersection(*map(set, sets))
    # now we can just return one of the common vertices to place the guard
    return pick_one(common)

lets test

In [17]:
def run_reduction_algorithm(H_star_hat, clique_contents):
    """
    PAPER OUTPUTS:
      k = guard count
      m = movement number
    Stop when only one hyperedge remains.
    Prioritize single-child deletions over multi-child deletions.
    """
    k = 0 # initial guard count
    m = 1 # initial movement count
    placements = [] # list of guard placements for debugging/analysis
    # Ensure we start in the proper "working form"
    H = minimize_by_inclusion(H_star_hat)
    H = remove_size1_edges(H)

    history = []

    while True:
        # get the hyperedge family as a dict: e -> set(vertices in e)
        E_map = edge_sets(H)

        # Stop condition: one hyperedge left
        if len(E_map) <= 1:
            break

        Ei, case, children, parents = pick_good_hyperedge_to_delete(H, prefer_single_child=True)

        # deterimining guard location
        if case == "single":
            leaf = children[0]
            loc = single_child_guard_location(leaf, parents, clique_contents)
            placements.append({
            "type": "single",
            "deleted_edge": Ei,
            "leaf": leaf,
            "parents": parents,
            "new_guard_at": loc
            })
        elif case == "multi":
            g1 = multi_child_guard_location(children, clique_contents)
            # g2 = support_guard_location_for_parents(g1, parents, clique_contents)
            placements.append({
                "type": "multi",
                "deleted_edge": Ei,
                "children": children,
                "parents": parents,
                "new_guard_at": g1,        # g1
                # "support_guard_at": g2     # g2 (not a new guard, but needed for explanation)
            })
        if Ei is None:
            # No "good" deletion exists under our connectivity rule.
            # the graph is not helly, or we have a bug in our code, but we stop safely instead of crashing.
            history.append({"stopped": True, "reason": "no good cut available", "edges": list(E_map.keys())})
            break

        # Count guards and movement according to your rules
        k += 1 # default guard cost for any deletion

        # Execute
        H, clique_contents = execute_delete_step(H, clique_contents, Ei, case, children, parents)

        history.append({
            "deleted_edge": Ei,
            "case": case,
            "children": children,
            "parents": parents,
            "k": k,
            "m": m,
            'placement': placements[-1]["new_guard_at"],
            "remaining_edges": list(edge_sets(H).keys())
        })

    # Base case add-ons (if we ended with exactly one hyperedge)
    E_map = edge_sets(H)
    if len(E_map) == 1:
        # clean up the last hyperedge case with cluster control add-ons
        H = base_case_cluster_control(H, clique_contents)

        # re-read the cleaned-up last edge and its vertices
        E_map = edge_sets(H)
        last_edge = next(iter(E_map))
        # get the number of vertices in the last remaining hyperedge
        n = len(E_map[last_edge])

        # guards increment by 1 if n=1 else 2 (per your rule for the base case)
        k += 1 if n == 1 else 2 

        # movement increment by 1 if n<3 else 2 (per your rule for the base case)
        m += 0 if n < 3 else 1

        # guard placement for the base case: we can just pick one of the vertices in the last edge to place the guard,
        # since the guard can see the whole last edge, it doesn't matter which one we pick for the base case

        # if the base case adds only one guard then we pick one of the two cliques in the last hyperedge.
        
        placements.append({
            "type": "base_case",
            "last_edge": last_edge,
            "n": n,
            "k_added": 1 if n == 1 else 2,
            "m_added": 1 if n < 3 else 2,
            # if 2 guards are added then new_guard_at will need to be a tuple of two vertices
            "new_guard_at": pick_one(E_map[last_edge]) if n < 2 else tuple(sorted(E_map[last_edge], key=str)[:2])
        })

        #
        history.append({"base_case_edge": last_edge, "n": n, "k_final": k, "m_final": m, "final_placement": placements[-1]["new_guard_at"] if placements else None})

    return k, m, H, clique_contents, history, placements

In [18]:
# Ĥ* hyperedges (edge -> clique-vertices)
# A is an α-leaf (only in E1)
# B is a parent initially (in E1 and E2)
# C is also a parent (in E1, E2, E3)
E_star_hat = {
    "E1": {"A", "B", "C", "D"},   # single-child edge because only A has deg=1
    "E2": {"D","E"},
    "E3": {"E","F","G","H"}
}

# clique_contents: what each clique-vertex "contains" in virtual G
# A shares v2 with B. After reduction, B loses v2 and becomes {v3}.
# C contains {v2,v3,v4}, so {v3} ⊆ C(C) and cluster control removes B.
clique_contents = {
    "A": {"v1", "v4"},
    "B": {"v2", "v4"},
    "C": {"v3", "v4"},
    "D": {"v4", "v5"},
    "E": {"v5", "v6"},
    "F": {"V6", "v7"},
    "G": {"v6", "v8"},
    "H": {"v6", "v9"}
}

H = build_hypergraph(E_star_hat)
H = minimize_by_inclusion(H)
H = remove_size1_edges(H)

print_hyperedges(H, "Initial Ĥ*:")
alpha, parents, deg = alpha_leaves_and_parents(H)
print("α-leaves:", sorted(alpha, key=str))
print("parents :", sorted(parents, key=str))
print("degrees :", dict(sorted(deg.items(), key=lambda kv: str(kv[0]))))

Initial Ĥ*:
  E1: ['A', 'B', 'C', 'D']
  E2: ['D', 'E']
  E3: ['E', 'F', 'G', 'H']
α-leaves: ['A', 'B', 'C', 'F', 'G', 'H']
parents : ['D', 'E']
degrees : {'A': 1, 'B': 1, 'C': 1, 'D': 2, 'E': 2, 'F': 1, 'G': 1, 'H': 1}


In [19]:
Ei, case, children, pars = pick_good_hyperedge_to_delete(H, prefer_single_child=True)
print("Chosen edge to delete:", Ei, "| case:", case)
print("Children in Ei:", children)
print("Parents in Ei :", pars)

H2, cc2 = execute_delete_step(H, clique_contents, Ei, case, children, pars)

print("\nAfter deleting Ei and applying reductions + cluster control:")
print_hyperedges(H2, "Ĥ* after step 1:")

alpha2, parents2, deg2 = alpha_leaves_and_parents(H2)
print("α-leaves:", sorted(alpha2, key=str))
print("parents :", sorted(parents2, key=str))
print("degrees :", dict(sorted(deg2.items(), key=lambda kv: str(kv[0]))))

print("\nClique contents after step 1:")
for k in sorted(cc2.keys(), key=str):
    print(k, ":", sorted(cc2[k]))

Chosen edge to delete: E1 | case: multi
Children in Ei: ['A', 'B', 'C']
Parents in Ei : ['D']

After deleting Ei and applying reductions + cluster control:
Ĥ* after step 1:
  E2: ['D', 'E']
  E3: ['E', 'F', 'G', 'H']
α-leaves: ['D', 'F', 'G', 'H']
parents : ['E']
degrees : {'D': 1, 'E': 2, 'F': 1, 'G': 1, 'H': 1}

Clique contents after step 1:
A : ['v1', 'v4']
B : ['v2', 'v4']
C : ['v3', 'v4']
D : ['v4', 'v5']
E : ['v5', 'v6']
F : ['V6', 'v7']
G : ['v6', 'v8']
H : ['v6', 'v9']


In [20]:
k, m, H_final, cc_final, history, placements = run_reduction_algorithm(H, clique_contents)

print("\n=== FINAL RESULTS ===")
print("Guard count k =", k)
print("Movement number m =", m)
print_hyperedges(H_final, "Final hypergraph:")

print("\n=== HISTORY ===")
for i, h in enumerate(history, 1):
    print(f"{i}. {h}")

# print("\n=== PLACEMENTS ===")
# for i, p in enumerate(placements, 1):
#     print(f"{i}. {p}")


=== FINAL RESULTS ===
Guard count k = 4
Movement number m = 2
Final hypergraph:
  E3: ['F', 'G', 'H']

=== HISTORY ===
1. {'deleted_edge': 'E1', 'case': 'multi', 'children': ['A', 'B', 'C'], 'parents': ['D'], 'k': 1, 'm': 1, 'placement': 'v4', 'remaining_edges': ['E2', 'E3']}
2. {'deleted_edge': 'E2', 'case': 'single', 'children': ['D'], 'parents': ['E'], 'k': 2, 'm': 1, 'placement': 'v5', 'remaining_edges': ['E3']}
3. {'base_case_edge': 'E3', 'n': 3, 'k_final': 4, 'm_final': 2, 'final_placement': ('F', 'G')}


RECURSION TESTING

We need a version of our pick_goof_hyperedge function to handle colonies

In [21]:
def pick_good_hyperedge_to_delete_case(H, case="prefer_single"):
    """
    case:
      - "prefer_single": try single-child good deletions first, else multi-child
      - "prefer_multi":  try multi-child good deletions first, else single-child
    Returns (Ei, case, children, parents) like your existing picker.
    """
    # use your existing function but with a twist:
    if case == "prefer_single":
        return pick_good_hyperedge_to_delete(H, prefer_single_child=True)
    else:
        # We need "prefer multi" behavior.
        # Easiest: call your picker twice by temporarily flipping preference:
        #   1) try to find a good multi by scanning candidates ourselves
        #   2) else fall back to prefer_single
        E_map = edge_sets(H)
        if len(E_map) <= 1:
            return None, None, None, None

        alpha, parents_set, children_in_edge, parents_in_edge = classify_hyperedges_by_children(H)

        def is_good_delete(Ei):
            E_after = {e: set(vs) for e, vs in E_map.items() if e != Ei}
            return is_connected_hyperedge_graph(E_after)

        multi_candidates = sorted([e for e in E_map if len(children_in_edge[e]) >= 2], key=str)
        for e in multi_candidates:
            if is_good_delete(e):
                return e, "multi", children_in_edge[e], parents_in_edge[e]

        # if no good multi exists, fall back to your standard preference
        return pick_good_hyperedge_to_delete(H, prefer_single_child=True)

In [22]:
def pick_good_in_region(H, region_edges, prefer_single=True):
    """
    Restrict candidate Ei to region_edges.
    prefer_single=True: try single-child first, else multi-child.
    Returns (Ei, case, children, parents) or (None,...)
    """
    E_map = edge_sets(H)
    region_edges = [e for e in region_edges if e in E_map]
    if len(region_edges) == 0:
        return None, None, None, None

    alpha, parents_set, children_in_edge, parents_in_edge = classify_hyperedges_by_children(H)

    def good(Ei):
        E_after = {e: set(vs) for e, vs in E_map.items() if e != Ei}
        return is_connected_hyperedge_graph(E_after)

    singles = sorted([e for e in region_edges if len(children_in_edge[e]) == 1], key=str)
    multis  = sorted([e for e in region_edges if len(children_in_edge[e]) >= 2], key=str)

    if prefer_single:
        for e in singles:
            if good(e):
                return e, "single", children_in_edge[e], parents_in_edge[e]
        for e in multis:
            if good(e):
                return e, "multi", children_in_edge[e], parents_in_edge[e]
    else:
        for e in multis:
            if good(e):
                return e, "multi", children_in_edge[e], parents_in_edge[e]
        for e in singles:
            if good(e):
                return e, "single", children_in_edge[e], parents_in_edge[e]

    return None, None, None, None

In [23]:
def base_case_score(H, clique_contents):
    #the H provided is highly reduced to base case, last hyperedge
    """
    Apply base-case cluster control, then return (k_add, m_add, base_placement_info).
    Uses your current 'm baseline = 1' convention:
      - if n < 3: m_add = 0
      - else:     m_add = 1
    """
    #clean the last hyperedge 
    H = base_case_cluster_control(H, clique_contents)

    E_map = edge_sets(H)
    if len(E_map) != 1:
        # safety: shouldn't happen
        return 0, 0, {"warning": "hyperedge is not in base case"}

    last_edge = next(iter(E_map))
    n = len(E_map[last_edge])

    k_add = 1 if n == 1 else 2
    m_add = 0 if n < 3 else 1

    #used from history in orig algorithm
    placement = pick_one(E_map[last_edge]) if n < 2 else tuple(sorted(E_map[last_edge], key=str)[:2])
    
    return k_add, m_add, {
        "type": "base_case",
        "last_edge": E_map[last_edge],
        "n": n,
        "k_added": k_add,
        "m_added": (1 if n < 3 else 2),
        "new_guard_at": placement
    }


In [24]:
def component_of_edge(E_map, anchor_edge):
    """Return the connected component (set of hyperedges) containing anchor_edge."""
    if anchor_edge not in E_map:
        return set()
    adj = hyperedge_adjacency(E_map)
    seen = {anchor_edge}
    stack = [anchor_edge]
    while stack:
        cur = stack.pop()
        for nb in adj[cur]:
            if nb not in seen:
                seen.add(nb)
                stack.append(nb)
    return seen


In [28]:
def colony_m_increment(s):
    """
    Colony size s.
    Smallest colony that forces m=2 is s=4.
    Extra movement beyond that grows linearly.
    """
    return max(0, s - 4)

Let do the recursion now

In [32]:
def solve_recursive_colonies(H_star_hat, clique_contents, depth=0):
    """
    Returns:
      k, m, H_final, clique_contents_final, history, placements, components

    components:
      {"type":"leaf", "members":[leaf]}
      {"type":"colony", "members":[...]}   # connected colony
    """
    indent = "  " * depth
    print("ENTRY edges:", edge_sets(H_star_hat))
    H = minimize_by_inclusion(H_star_hat)
    H = remove_size1_edges(H)

    history = []
    placements = []
    components = []

    #if empty graph
    E_map = edge_sets(H)
    if len(E_map) == 0:
        return 0, 1, H, clique_contents, history, placements, components

    #base case
    if len(E_map) == 1:
        k_add, m_add, base_place = base_case_score(H, clique_contents)
        k = k_add
        m = m_add + 1 if base_place["n"] >=3 else 0
        placements.append(base_place)
        history.append({"base_case": True,
                        "n": base_place["n"], 
                        "k": k, 
                        "m": m,})
        return k, m, H, clique_contents, history, placements, components

    colony = [] # members (children removed by multi) + terminal single leaf
    colony_region = None # set of hyperedges allowed during the colony run

    while True:
        E_map = edge_sets(H)
        if len(E_map) <= 1:
            return solve_recursive_colonies(H, clique_contents, depth=depth+1)

        if len(colony) == 0:
            # normal phase: pick globally, prefer single
            Ei, case, children, parents = pick_good_hyperedge_to_delete_case(H, case="prefer_single")

            if Ei is None:
                history.append({"stopped": True, "reason": "no good cut available", "edges": list(E_map.keys())})
                return solve_recursive_colonies(H, clique_contents, depth=depth+1)

            # if we START a colony with a multi cut, lock the region
            if case == "multi":
                colony_region = component_of_edge(E_map, Ei)

        else:
            # colony active: DO NOT jump across the hypergraph.
            # First we try to CLOSE the colony with a single-child cut inside the same region.
            Ei, case, children, parents = pick_good_in_region(H, colony_region, prefer_single=True)

            if Ei is None:
                # If no single works in the region, then try multi in the region.
                Ei, case, children, parents = pick_good_in_region(H, colony_region, prefer_single=False)

            if Ei is None:
                history.append({"stopped": True, "reason": "no good cut inside colony region", "region": list(colony_region)})
                return solve_recursive_colonies(H, clique_contents, depth=depth+1)

        # from orig algorithm
        # if there is another R1 move after the last R1 move:
        if case == "multi":
            g1 = multi_child_guard_location(children, clique_contents)
            placements.append({
                "type": "multi",
                "deleted_edge": Ei,
                "children": children,
                "parents": parents,
                "new_guard_at": g1
            })

            H, clique_contents = execute_delete_step(H, clique_contents, Ei, case, children, parents)
            colony.extend(children) #add leaves to the colony

            history.append({
                "deleted_edge": Ei,
                "case": "multi",
                "children": children,
                "parents": parents,
                "colony_size": len(colony),
                "region_size": len(colony_region) if colony_region is not None else None,
                "remaining_edges": list(edge_sets(H).keys())
            })

            # after H changes
            if colony_region is not None:
                colony_region = set([e for e in colony_region if e in edge_sets(H)])

        # SINGLE case from orig algorithm
        # if an R2 move is possible, then we do that and close the colony:
        if case == "single":
            leaf = children[0]
            loc = single_child_guard_location(leaf, parents, clique_contents)
            placements.append({
                "type": "single",
                "deleted_edge": Ei,
                "leaf": leaf,
                "parents": parents,
                "new_guard_at": loc
            })

            H, clique_contents = execute_delete_step(H, clique_contents, Ei, case, children, parents)

            history.append({
                "deleted_edge": Ei,
                "case": "single",
                "leaf": leaf,
                "parents": parents,
                "remaining_edges": list(edge_sets(H).keys())
            })

            # component
            if len(colony) == 0: #we did a single child case
                comp = {"type": "leaf", "members": [leaf]}
            else: #we found a colony
                comp = {"type": "colony", "members": colony + [leaf]}
            components.append(comp)

            # recurse on reduced base
            k_base, m_base, H_final, cc_final, hist2, plac2, comps2 = solve_recursive_colonies(
                H, clique_contents, depth=depth+1
            )

            # final counting
            #if the previous step was R2, we just add one more guard
            if comp["type"] == "leaf":
                k = k_base + 1 
                m = 1
            else: #if we have to add back a colony then
                # we need 2 guards and movement number
                s = len(comp["members"])
                k = k_base + 2
                m = m = max(m_base, 2) + colony_m_increment(s)

            history.extend(hist2)
            placements.extend(plac2)
            components.extend(comps2)

            return k, m, H_final, cc_final, history, placements, components

test the recursion

In [33]:
# Ĥ* hyperedges (edge -> clique-vertices)
# A is an α-leaf (only in E1)
# B is a parent initially (in E1 and E2)
# C is also a parent (in E1, E2, E3)
E_star_hat = {
    "E1": {"A", "B", "C", "D"},   # single-child edge because only A has deg=1
    "E2": {"D","E"},
    "E3": {"E","F","G","H"}
}

# clique_contents: what each clique-vertex "contains" in virtual G
# A shares v2 with B. After reduction, B loses v2 and becomes {v3}.
# C contains {v2,v3,v4}, so {v3} ⊆ C(C) and cluster control removes B.
clique_contents = {
    "A": {"v1", "v4"},
    "B": {"v2", "v4"},
    "C": {"v3", "v4"},
    "D": {"v4", "v5"},
    "E": {"v5", "v6"},
    "F": {"v6", "v7"},
    "G": {"v6", "v8"},
    "H": {"v6", "v9"}
}


H = build_hypergraph(E_star_hat)
print("Before minimize:", edge_sets(H))
H = minimize_by_inclusion(H)
print("After minimize :", edge_sets(H))
H = remove_size1_edges(H)
print("After rm size1 :", edge_sets(H))

print_hyperedges(H, "Initial Ĥ*:")
alpha, parents, deg = alpha_leaves_and_parents(H)
print("α-leaves:", sorted(alpha, key=str))
print("parents :", sorted(parents, key=str))
print("degrees :", dict(sorted(deg.items(), key=lambda kv: str(kv[0]))))

Before minimize: {'E1': {'B', 'A', 'D', 'C'}, 'E2': {'E', 'D'}, 'E3': {'G', 'E', 'H', 'F'}}
After minimize : {'E1': {'B', 'A', 'D', 'C'}, 'E2': {'E', 'D'}, 'E3': {'G', 'E', 'H', 'F'}}
After rm size1 : {'E1': {'B', 'A', 'D', 'C'}, 'E2': {'E', 'D'}, 'E3': {'G', 'E', 'H', 'F'}}
Initial Ĥ*:
  E1: ['A', 'B', 'C', 'D']
  E2: ['D', 'E']
  E3: ['E', 'F', 'G', 'H']
α-leaves: ['A', 'B', 'C', 'F', 'G', 'H']
parents : ['D', 'E']
degrees : {'A': 1, 'B': 1, 'C': 1, 'D': 2, 'E': 2, 'F': 1, 'G': 1, 'H': 1}


In [34]:


k, m, H_final, cc_final, hist, plac, comps = solve_recursive_colonies(H, clique_contents)
print("Guards: ", k, "Movement: ", m)
print("Components: ", comps)
print("placements: ", plac)
print("History:", hist)

ENTRY edges: {'E1': {'B', 'A', 'D', 'C'}, 'E2': {'E', 'D'}, 'E3': {'G', 'E', 'H', 'F'}}
ENTRY edges: {'E3': {'G', 'H', 'F'}}
Guards:  4 Movement:  2
Components:  [{'type': 'colony', 'members': ['A', 'B', 'C', 'D']}]
placements:  [{'type': 'multi', 'deleted_edge': 'E1', 'children': ['A', 'B', 'C'], 'parents': ['D'], 'new_guard_at': 'v4'}, {'type': 'single', 'deleted_edge': 'E2', 'leaf': 'D', 'parents': ['E'], 'new_guard_at': 'v5'}, {'type': 'base_case', 'last_edge': {'G', 'H', 'F'}, 'n': 3, 'k_added': 2, 'm_added': 2, 'new_guard_at': ('F', 'G')}]
History: [{'deleted_edge': 'E1', 'case': 'multi', 'children': ['A', 'B', 'C'], 'parents': ['D'], 'colony_size': 3, 'region_size': 3, 'remaining_edges': ['E2', 'E3']}, {'deleted_edge': 'E2', 'case': 'single', 'leaf': 'D', 'parents': ['E'], 'remaining_edges': ['E3']}, {'base_case': True, 'n': 3, 'k': 2, 'm': 2}]
